In [3]:
import pandas as pd
import re
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# used for reducing words to base level
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# read in data
df = pd.read_csv('FakeNewsNet_scraped.csv', nrows=1000)

# clean function
def clean_text(text):
    # removes noise from data using regular expressions
    if pd.isna(text): return ''
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    # convert words to base dictionary meaning
    tokens = [lemmatizer.lemmatize(w) for w in text.split()
              if w not in stop_words and len(w) > 2]
    return ' '.join(tokens)

# url data cleaning
def engineer_url_features(df):
    # deals with missing values with blank spaces (TODO: could implement better logic for na values)
    url  = df['news_url'].fillna('')
    furl = df['fixed_url'].fillna('')
    # extract more useful features from url's
    return pd.DataFrame({
        'url_length':       url.str.len(),
        'url_num_slashes':  url.str.count('/'),
        'url_has_https':    url.str.startswith('https').astype(int),
        'url_suspicious':   url.str.contains(
                                r'breaking|alert|shock|secret|viral|hoax',
                                case=False, na=False).astype(int),
        'url_was_redirected': (url != furl).astype(int),
    })

# clean both the title and body of the article
df['clean_title'] = df['title'].apply(clean_text)
df['clean_body']  = df['scraped_words'].apply(clean_text)
# collect info from urls
url_feats = engineer_url_features(df)
df = pd.concat([df, url_feats], axis=1)

# Fill in missing source_domain if exists
df['source_domain'] = df['source_domain'].fillna('unknown')

numeric_cols = [
    'tweet_num', 'scraped_word_count',
    'url_length', 'url_num_slashes',
    'url_has_https', 'url_suspicious', 'url_was_redirected'
]
categorical_cols = ['source_domain']
text_title_col   = 'clean_title'
text_body_col    = 'clean_body'

feature_cols = [text_title_col, text_body_col] + numeric_cols + categorical_cols
X = df[feature_cols]
y = df['real']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

preprocessor = ColumnTransformer(transformers=[
    # attempted both 10,000 and 5,000
    # best result with more features for title as it contains bulk of patterns
    ('title', TfidfVectorizer(
        max_features=15000, ngram_range=(1, 2),
        sublinear_tf=True), text_title_col),
    # tried 20,000 and 35,000 as well, best result with 40,000 due to logic above
    ('body', TfidfVectorizer(
        max_features=40000, ngram_range=(1, 2),
        sublinear_tf=True, min_df=2), text_body_col),
    # standardize numerical columns
    ('num', StandardScaler(), numeric_cols),
    # on hot encode categoric columns
    ('cat', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False), categorical_cols),
],
remainder='drop')

# build pipeline
pipeline = Pipeline([
    ('features', preprocessor),
    ('clf', LogisticRegression(
        max_iter=1000, class_weight='balanced', C=1.0)),
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Fake', 'Real']))

              precision    recall  f1-score   support

        Fake       0.96      0.96      0.96        48
        Real       0.99      0.99      0.99       152

    accuracy                           0.98       200
   macro avg       0.97      0.97      0.97       200
weighted avg       0.98      0.98      0.98       200



In [4]:
# Save the model
import joblib


model_filename = "fake_news_model_final_version.pkl"
print(f"Saving the model to {model_filename}...")
joblib.dump(pipeline, model_filename)
print("Done!")

Saving the model to fake_news_model_final_version.pkl...
Done!
